In [161]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [172]:
# ─────────────────────────────────────────────────────────
# CELL 1 — Setup
# ─────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import os
import joblib
import gc
from sklearn.preprocessing import MinMaxScaler

RAW_FOLDER  = '/content/drive/MyDrive/EV_PROJECT/Raw_Trips'
SAVE_FOLDER = '/content/drive/MyDrive/EV_PROJECT/Ready_for_LSTM_v11'
os.makedirs(SAVE_FOLDER, exist_ok=True)

csv_files = sorted([
    f for f in os.listdir(RAW_FOLDER)
    if f.endswith('.csv')
])
print(f"Found {len(csv_files)} trip files")

Found 70 trip files


In [173]:
# ─────────────────────────────────────────────────────────
# CELL 2 — Constants and Feature Definitions
# ─────────────────────────────────────────────────────────

# SoC removed from features — it is now the TARGET
# Power_kW and Road_Gradient_deg are engineered features
FEATURE_COLS = [
    # Driving dynamics
    'Velocity [km/h]',
    'Throttle [%]',
    'Motor Torque [Nm]',
    'Regenerative Braking Signal',
    # Battery
    'Battery Voltage [V]',
    'Battery Temperature [°C]',
    # HVAC
    'Heating Power CAN [kW]',
    'Requested Heating Power [W]',
    'AirCon Power [kW]',
    'Heater Signal',
    'Heat Exchanger Temperature [°C]',
    'Cabin Temperature Sensor [°C]',
    # Environment
    'Ambient Temperature [°C]',
    'Elevation [m]',
    # Engineered
    'Road_Gradient_deg',
    'Power_kW'
]

# Target — SoC normalised to 0-1
# RDR in km is derived at prediction time as:
# RDR_km = (SoC_fraction × BATTERY_KWH) / AVG_EFFICIENCY
TARGET_COL     = 'SoC_fraction'
TIME_COL       = 'Time [s]'

BATTERY_KWH    = 18.8    # BMW i3 usable capacity
AVG_EFFICIENCY = 0.15    # kWh/km — BMW i3 typical

WINDOW         = 60      # 60 seconds context at 1 Hz
STRIDE         = 1       # dense windowing
DOWNSAMPLE     = 5       # 10 Hz → 1 Hz

print(f"Feature count  : {len(FEATURE_COLS)}")
print(f"Target         : {TARGET_COL} (SoC/100)")
print(f"Window         : {WINDOW}s at 1 Hz")
print(f"Stride         : {STRIDE}")
print(f"RDR conversion : (SoC × {BATTERY_KWH}) / {AVG_EFFICIENCY}")

Feature count  : 16
Target         : SoC_fraction (SoC/100)
Window         : 60s at 1 Hz
Stride         : 1
RDR conversion : (SoC × 18.8) / 0.15


In [174]:
# ─────────────────────────────────────────────────────────
# CELL 3 — Single Trip Loader
# ─────────────────────────────────────────────────────────

def load_trip(filename):
    """
    Load one raw trip CSV.
    Target is SoC_fraction = SoC[%] / 100
    Returns (df, available_features) or None if unusable.
    """
    path = os.path.join(RAW_FOLDER, filename)

    with open(path, 'r', encoding='latin-1') as fh:
        first_line = fh.readline()
    sep = ';' if first_line.count(';') > first_line.count(',') else ','

    df = pd.read_csv(path, encoding='latin-1', sep=sep)
    df.columns = df.columns.str.strip()

    # Must have these columns
    if 'Velocity [km/h]' not in df.columns:
        print(f"  SKIP {filename} — Velocity missing")
        return None
    if TIME_COL not in df.columns:
        print(f"  SKIP {filename} — Time column missing")
        return None
    if 'SoC [%]' not in df.columns:
        print(f"  SKIP {filename} — SoC missing")
        return None

    # ── Downsample to 1 Hz ───────────────────────────────
    df = df.iloc[::DOWNSAMPLE].reset_index(drop=True)

    # ── Time delta ────────────────────────────────────────
    dt = df[TIME_COL].diff().fillna(1.0)

    # ── Engineer: Road Gradient ───────────────────────────
    dx_m   = (df['Velocity [km/h]'] / 3.6) * dt
    d_elev = df['Elevation [m]'].diff().fillna(0) if 'Elevation [m]' in df.columns else 0
    df['Road_Gradient_deg'] = np.degrees(
        np.arctan2(d_elev, dx_m.replace(0, np.nan))
    ).fillna(0)

    # ── Engineer: Power ───────────────────────────────────
    df['Power_kW'] = (
        df['Battery Voltage [V]'] * df['Battery Current [A]']
    ) / 1000

    # ── Target: SoC fraction (0 to 1) ────────────────────
    df['SoC_fraction'] = (df['SoC [%]'] / 100.0).clip(0, 1)

    # ── Select final columns ──────────────────────────────
    available_features = [c for c in FEATURE_COLS if c in df.columns]
    missing_features   = [c for c in FEATURE_COLS if c not in df.columns]

    if missing_features:
        print(f"  NOTE {filename} — missing: {missing_features}")

    df = df[available_features + ['SoC_fraction']].copy()

    # ── Interpolate and clean ─────────────────────────────
    df = df.interpolate(method='linear', limit=3)
    df = df.dropna().reset_index(drop=True)

    if len(df) < WINDOW + 1:
        print(f"  SKIP {filename} — too short ({len(df)} rows)")
        return None

    return df, available_features


# ── Test on one trip ──────────────────────────────────────
result = load_trip(csv_files[0])
if result is not None:
    df_test, feat_test = result
    print(f"\nTest trip : {csv_files[0]}")
    print(f"  Shape   : {df_test.shape}")
    print(f"  SoC range: {df_test['SoC_fraction'].min():.3f} → {df_test['SoC_fraction'].max():.3f}")
    print(f"  Features: {len(feat_test)}/{len(FEATURE_COLS)}")


Test trip : TripA01.csv
  Shape   : (2018, 17)
  SoC range: 0.815 → 0.869
  Features: 16/16


In [175]:
# ─────────────────────────────────────────────────────────
# CELL 4 — Fit Global MinMaxScaler on features only
# ─────────────────────────────────────────────────────────
scaler  = MinMaxScaler()
skipped = []

print("Fitting global scaler across all trips...\n")

for f in csv_files:
    result = load_trip(f)
    if result is None:
        skipped.append(f)
        continue
    df, available_features = result
    scaler.partial_fit(df[available_features])
    del df
    gc.collect()

scaler_path = os.path.join(SAVE_FOLDER, 'global_scaler.pkl')
joblib.dump(scaler, scaler_path)

print(f"Scaler fitted and saved → {scaler_path}")
print(f"Trips processed : {len(csv_files) - len(skipped)}/70")
if skipped:
    print(f"Skipped         : {skipped}")

Fitting global scaler across all trips...

Scaler fitted and saved → /content/drive/MyDrive/EV_PROJECT/Ready_for_LSTM_v11/global_scaler.pkl
Trips processed : 70/70


In [176]:
# ─────────────────────────────────────────────────────────
# CELL 5 — Client Trip Assignments (unchanged)
# ─────────────────────────────────────────────────────────
a_trips = [f for f in csv_files if f.startswith('TripA')]
b_trips = [f for f in csv_files if f.startswith('TripB')]

b1 = b_trips[0:13]
b2 = b_trips[13:26]
b3 = b_trips[26:]

clients = {
    'client_A':  a_trips,
    'client_B1': b1,
    'client_B2': b2,
    'client_B3': b3,
}

print("Client assignments:")
for name, trips in clients.items():
    n_train = max(1, int(len(trips) * 0.8))
    n_test  = len(trips) - n_train
    print(f"  {name}: {len(trips)} trips → {n_train} train | {n_test} test")

Client assignments:
  client_A: 32 trips → 25 train | 7 test
  client_B1: 13 trips → 10 train | 3 test
  client_B2: 13 trips → 10 train | 3 test
  client_B3: 12 trips → 9 train | 3 test


In [177]:
# ─────────────────────────────────────────────────────────
# CELL 6 — Build Windowed .npy Datasets Per Client
# ─────────────────────────────────────────────────────────

def build_client_data(client_name, trip_files):

    X_train_list    = []
    y_train_list    = []   # SoC fraction
    X_test_list     = []
    y_test_list     = []   # SoC fraction
    y_test_km_list  = []   # RDR in km (for evaluation)

    split_idx   = max(1, int(len(trip_files) * 0.8))
    train_trips = trip_files[:split_idx]
    test_trips  = trip_files[split_idx:]

    print(f"\n{'='*55}")
    print(f"{client_name} | {len(train_trips)} train | {len(test_trips)} test")
    print('='*55)

    # ── TRAIN ─────────────────────────────────────────────
    print("\n  --- TRAIN ---")
    for f in train_trips:
        result = load_trip(f)
        if result is None:
            continue

        df, available_features = result
        X_scaled = scaler.transform(df[available_features].values)

        # Pad to full feature count if any missing
        if len(available_features) < len(FEATURE_COLS):
            X_full   = np.zeros((len(df), len(FEATURE_COLS)), dtype=np.float32)
            feat_idx = [FEATURE_COLS.index(c) for c in available_features]
            X_full[:, feat_idx] = X_scaled
            X_scaled = X_full

        y_soc     = df['SoC_fraction'].values
        n_windows = 0

        for i in range(0, len(df) - WINDOW, STRIDE):
            X_train_list.append(X_scaled[i : i + WINDOW])
            y_train_list.append(y_soc[i + WINDOW])
            n_windows += 1

        print(f"  {f}: {len(df):5d} rows → {n_windows:5d} windows")
        del df
        gc.collect()

    # ── TEST ──────────────────────────────────────────────
    print("\n  --- TEST ---")
    for f in test_trips:
        result = load_trip(f)
        if result is None:
            continue

        df, available_features = result
        X_scaled = scaler.transform(df[available_features].values)

        if len(available_features) < len(FEATURE_COLS):
            X_full   = np.zeros((len(df), len(FEATURE_COLS)), dtype=np.float32)
            feat_idx = [FEATURE_COLS.index(c) for c in available_features]
            X_full[:, feat_idx] = X_scaled
            X_scaled = X_full

        y_soc     = df['SoC_fraction'].values
        n_windows = 0

        for i in range(0, len(df) - WINDOW, STRIDE):
            X_test_list.append(X_scaled[i : i + WINDOW])
            y_test_list.append(y_soc[i + WINDOW])
            # Convert SoC to RDR km for evaluation
            rdr_km = (y_soc[i + WINDOW] * BATTERY_KWH) / AVG_EFFICIENCY
            y_test_km_list.append(rdr_km)
            n_windows += 1

        print(f"  {f}: {len(df):5d} rows → {n_windows:5d} windows")
        del df
        gc.collect()

    # ── Stack and save ────────────────────────────────────
    X_tr       = np.array(X_train_list,   dtype=np.float32)
    y_tr       = np.array(y_train_list,   dtype=np.float32)
    X_te       = np.array(X_test_list,    dtype=np.float32)
    y_te       = np.array(y_test_list,    dtype=np.float32)
    y_te_km    = np.array(y_test_km_list, dtype=np.float32)

    client_dir = os.path.join(SAVE_FOLDER, client_name)
    os.makedirs(client_dir, exist_ok=True)

    np.save(os.path.join(client_dir, 'X_train.npy'),   X_tr)
    np.save(os.path.join(client_dir, 'y_train.npy'),   y_tr)
    np.save(os.path.join(client_dir, 'X_test.npy'),    X_te)
    np.save(os.path.join(client_dir, 'y_test.npy'),    y_te)
    np.save(os.path.join(client_dir, 'y_test_km.npy'), y_te_km)

    print(f"\n  SAVED {client_name}:")
    print(f"    X_train  : {X_tr.shape}")
    print(f"    y_train  : {y_tr.shape}  SoC range {y_tr.min():.3f}→{y_tr.max():.3f}")
    print(f"    X_test   : {X_te.shape}")
    print(f"    y_test   : {y_te.shape}  SoC range {y_te.min():.3f}→{y_te.max():.3f}")
    print(f"    y_test_km: {y_te_km.shape}  RDR range {y_te_km.min():.1f}→{y_te_km.max():.1f} km")

    del X_tr, y_tr, X_te, y_te, y_te_km
    gc.collect()


for client_name, trip_files in clients.items():
    build_client_data(client_name, trip_files)

print("\n✅ All client datasets saved!")


client_A | 25 train | 7 test

  --- TRAIN ---
  TripA01.csv:  2018 rows →  1958 windows
  TripA02.csv:  2826 rows →  2766 windows


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


  TripA03.csv:  1342 rows →  1282 windows
  TripA04.csv:   825 rows →   765 windows
  TripA05.csv:  2734 rows →  2674 windows


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


  TripA06.csv:  6329 rows →  6269 windows
  TripA07.csv:  4187 rows →  4127 windows


...

  TripB36.csv:  5705 rows →  5645 windows


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


  TripB37.csv:  2835 rows →  2775 windows
  TripB38.csv:  3286 rows →  3226 windows


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(



  SAVED client_B3:
    X_train  : (16787, 60, 16)
    y_train  : (16787,)  SoC range 0.154→0.852
    X_test   : (11646, 60, 16)
    y_test   : (11646,)  SoC range 0.444→0.836
    y_test_km: (11646,)  RDR range 55.6→104.8 km

✅ All client datasets saved!


In [178]:
# ─────────────────────────────────────────────────────────
# CELL 7 — Verification
# ─────────────────────────────────────────────────────────
print("="*55)
print("FINAL DATASET SUMMARY")
print("="*55)

for client in clients.keys():
    client_dir = os.path.join(SAVE_FOLDER, client)
    X_tr    = np.load(os.path.join(client_dir, 'X_train.npy'), mmap_mode='r')
    y_tr    = np.load(os.path.join(client_dir, 'y_train.npy'), mmap_mode='r')
    X_te    = np.load(os.path.join(client_dir, 'X_test.npy'),  mmap_mode='r')
    y_te    = np.load(os.path.join(client_dir, 'y_test.npy'),  mmap_mode='r')
    y_te_km = np.load(os.path.join(client_dir, 'y_test_km.npy'), mmap_mode='r')

    print(f"\n{client}:")
    print(f"  X_train   : {X_tr.shape}")
    print(f"  y_train   : SoC {y_tr.min():.3f} → {y_tr.max():.3f}")
    print(f"  X_test    : {X_te.shape}")
    print(f"  y_test    : SoC {y_te.min():.3f} → {y_te.max():.3f}")
    print(f"  y_test_km : RDR {y_te_km.min():.1f} → {y_te_km.max():.1f} km")

    del X_tr, y_tr, X_te, y_te, y_te_km

gc.collect()
print("\n✅ Ready for LSTM training")

FINAL DATASET SUMMARY

client_A:
  X_train   : (70147, 60, 16)
  y_train   : SoC 0.342 → 0.884
  X_test    : (21486, 60, 16)
  y_test    : SoC 0.595 → 0.864
  y_test_km : RDR 74.6 → 108.3 km

client_B1:
  X_train   : (43679, 60, 16)
  y_train   : SoC 0.270 → 0.859
  X_test    : (8487, 60, 16)
  y_test    : SoC 0.306 → 0.733
  y_test_km : RDR 38.4 → 91.9 km

client_B2:
  X_train   : (30983, 60, 16)
  y_train   : SoC 0.346 → 0.855
  X_test    : (5430, 60, 16)
  y_test    : SoC 0.212 → 0.531
  y_test_km : RDR 26.6 → 66.6 km

client_B3:
  X_train   : (16787, 60, 16)
  y_train   : SoC 0.154 → 0.852
  X_test    : (11646, 60, 16)
  y_test    : SoC 0.444 → 0.836
  y_test_km : RDR 55.6 → 104.8 km

✅ Ready for LSTM training
